# PairwiseGP

この Notebook では、pairwise comparison から preference を学習する `robotorchan.models.PairwiseGP` を扱います。

通常の回帰と異なり、観測値はスカラー目的値ではなく「項目 i の方が項目 j より好ましい」といった比較結果です。

## このNotebookで学ぶこと

- **回帰**の `train_X, train_Y` と、**選好学習**の `datapoints, comparisons` の違い
- `[winner, loser]` 形式の比較データの生成
- 最新のrobotorchan公開API `PairwiseGP.fit()` と `posterior()` の使い方
- 潜在効用のPosteriorを可視化し、**比較結果から推定できる量**を理解する
- `AnalyticExpectedUtilityOfBestOption`（EUBO）で次に比較する候補を提案する
- 2候補の比較結果を履歴に追加して、モデルを再学習する

**注意：** 潜在効用の真値は合成例でのみ使います。実際の選好学習では、通常は絶対的な効用値を観測できません。

## 1. このモデルを使う場面

ユーザー・専門家・実験から絶対スコアより相対的な好みを自然かつ安定して取得できる場合に `PairwiseGP` を使います。人間の選好最適化、官能評価、ランキング、定量化しにくい材料スクリーニングなどが例です。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.preference import AnalyticExpectedUtilityOfBestOption
from botorch.optim import optimize_acqf

from robotorchan.models import PairwiseGP

torch.manual_seed(0)
dtype = torch.double

## 2. 潜在効用と比較データの生成

各候補点には直接観測しない潜在的な utility があるとし、その大小関係から比較データを生成します。`comparisons` の各行は `[winner, loser]` の順です。

### 比較データは「勝者・敗者」のインデックス

`datapoints` の各行が候補アイテムで、`comparisons[k] = [i, j]` は**アイテムiがアイテムjより好まれた**ことを表します。入力座標そのものではなく、`datapoints` の**行番号**を渡します。

この合成例は効用の大小で勝敗を決めるため、比較ノイズがありません。実際の人間の評価には揺らぎや矛盾が生じます。`PairwiseGP` の選好尤度は、比較結果を確率的に扱います。

In [ ]:
def latent_utility(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.0 * torch.pi * x).squeeze(-1) + 0.35 * x.squeeze(-1)

n_items = 18
datapoints = torch.linspace(0.0, 1.0, n_items, dtype=dtype).unsqueeze(-1)
utility = latent_utility(datapoints)

n_comparisons = 50
pairs = torch.randint(0, n_items, (n_comparisons, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]

comparisons = []
for i, j in pairs.tolist():
    # PairwiseGP では各行を [winner, loser] として与えます。
    if utility[i] >= utility[j]:
        comparisons.append([i, j])
    else:
        comparisons.append([j, i])

comparisons = torch.tensor(comparisons, dtype=torch.long)
datapoints.shape, comparisons.shape

In [ ]:
print("first 5 comparisons [winner, loser]:")
for winner, loser in comparisons[:5].tolist():
    print(f"item {winner} (x={datapoints[winner, 0]:.3f}) preferred over "
          f"item {loser} (x={datapoints[loser, 0]:.3f})")
assert comparisons.dtype == torch.long
assert comparisons.shape[-1] == 2
assert comparisons.min() >= 0 and comparisons.max() < len(datapoints)

## 3. モデル構築と robotorchan API

通常の `train_X / train_Y` ではなく `datapoints / comparisons` を渡します。robotorchan は `raw_datapoints` と `raw_comparisons` を保持します。

In [ ]:
model = PairwiseGP(
    datapoints=datapoints,
    comparisons=comparisons,
)

print("raw_datapoints:", model.raw_datapoints.shape)
print("raw_comparisons:", model.raw_comparisons.shape)
print("supports_mll:", model.supports_mll)

mll = model.make_mll()
type(mll).__name__

### 最新公開APIの確認

robotorchanの `PairwiseGP` はBoTorchのPairwiseGPをラップし、入力データを `raw_datapoints` / `raw_comparisons` に保持します。

`supports_fit=True` で、`model.fit()` が公開されています。内部では `make_mll()` で作る **Pairwise Laplace MLL** をBoTorchの最適化器で学習します。標準的なGaussian回帰のExact MLLとは異なります。

## 4. Preference model の学習

`PairwiseGP` は `SingleTaskGP` の Gaussian regression likelihood とは異なり、Laplace approximation を利用します。`make_mll()` は `PairwiseLaplaceMarginalLogLikelihood` を返します。

In [ ]:
mll = model.make_mll()
model.fit()
model.eval()

### なぜ潜在効用を学習するのか

観測されるのは比較の勝敗であり、`train_Y` のような効用の絶対値ではありません。PairwiseGPは、比較の勝敗を説明する**潜在効用関数**をGPで推定します。

効用の基準（特に加法定数）は比較から直接決まらないため、次の図では真の効用と予測平均を中心化して表示します。図の帯は潜在効用の近似的な事後標準偏差から計算したもので、勝敗確率の95%区間ではありません。

## 5. 潜在 utility の posterior

比較データだけから学習したモデルに対し、連続グリッド上で潜在 utility の posterior を取得します。

In [ ]:
test_X = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.sqrt().squeeze(-1)

# Pairwise utility は加法定数やスケールの規約を除いて識別されます。
# 可視化では両曲線を中心化して比較します。
true_u = latent_utility(test_X)
mean_centered = mean - mean.mean()
true_centered = true_u - true_u.mean()
lower = mean_centered - 1.96 * std
upper = mean_centered + 1.96 * std

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(test_X.squeeze(-1), true_centered, linestyle="--", label="true latent utility (centered)")
plt.plot(test_X.squeeze(-1), mean_centered, label="posterior mean (centered)")
plt.fill_between(test_X.squeeze(-1), lower, upper, alpha=0.2, label="95% interval")
plt.scatter(datapoints.squeeze(-1), utility - utility.mean(), s=25, alpha=0.5, label="items")
plt.xlabel("x")
plt.ylabel("relative utility")
plt.legend()
plt.title("PairwiseGP latent utility")
plt.show()

### 可視化の読み方

実線は比較データから推定した潜在効用の平均、破線は合成データの真の効用です。縦軸の絶対値より、**どの入力が相対的に好まれるか**を読み取ります。

帯が狭いことと、次の比較結果を確実に当てられることは同義ではありません。比較ノイズやモデルの仮定も影響します。

## 6. 現時点で好ましい領域を確認

学習には pairwise comparison しか使っていませんが、posterior mean を利用して候補を順位付けできます。ここでは posterior mean が最大となる位置を確認します。

In [ ]:
best_idx = mean.argmax()
best_x = test_X[best_idx]
print(f"posterior-best x: {best_x.item():.4f}")

### 事後平均の最大点は「現在の推定最良候補」

この候補は、学習済みモデルが最も好ましいと予測する点です。ただし**次に比較して情報を得るための候補**とは限りません。探索では、効用の不確実性も利用するEUBO獲得関数を使います。

## 7. このモデルを使わない方がよい場合

信頼できる絶対値のスカラー目的値を取得できるなら、`SingleTaskGP` などの通常回帰の方が単純で、1観測あたりの情報量も多いことが一般的です。`PairwiseGP` は測定・評価プロセス自体が比較形式である場合に特に有効です。

## Preference acquisition and candidate comparison

既知の比較対象を固定し、次に比較する新しい入力を獲得関数で最適化します。候補の評価は数値回帰値ではなく、どちらを好むかというペア比較です。

### EUBO：比較の価値を獲得関数にする

`AnalyticExpectedUtilityOfBestOption`（EUBO）は、比較後に得られる最良候補の効用を期待値として評価します。

ここでは `previous_winner=incumbent` によって比較対象を固定し、**新しい1候補**を `q=1` で最適化します。`optimize_acqf()` は候補の入力座標を最適化するもので、効用の真値は必要ありません。

`candidate_wins` は教育用の合成効用で勝敗を確認する処理です。実務では人間の評価者や実験の比較結果に置き換えます。

In [ ]:
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
incumbent = best_x.reshape(1, 1)
acquisition = AnalyticExpectedUtilityOfBestOption(
    pref_model=model,
    previous_winner=incumbent,
)
candidate, acquisition_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
assert candidate.shape == (1, 1)
assert torch.all((candidate >= bounds[0]) & (candidate <= bounds[1]))
candidate_wins = latent_utility(candidate) > latent_utility(incumbent)
print("candidate:", candidate)
print("acquisition value:", acquisition_value)
print("candidate preferred to incumbent:", bool(candidate_wins.item()))


## 8. EUBOで2候補を提案し、比較結果で更新

ここでは既知の勝者を固定せず、`q=2`で比較する2候補を同時に提案します。実際の評価者の代わりに、合成潜在効用を使って勝者を決定します。

### q=2で2候補を同時に提案する

`previous_winner` を指定しないEUBOでは、比較する2候補を同時に選ぶ `q=2` の構成を使用します。

反復の流れは次のとおりです。

1. 現在の比較履歴で `PairwiseGP` を構築し、`fit()` する
2. EUBOを `optimize_acqf(q=2)` で最大化し、2候補を得る
3. 2候補を実際に比較して勝者・敗者を決める
4. 候補を `history_X` の末尾に追加する
5. 新しい候補の**行番号**で `[winner, loser]` を作り、`history_comparisons` に追加する

この例では簡単のため各反復でモデルを再構築・再学習しています。

In [ ]:
torch.manual_seed(0)
history_X = datapoints.clone()
history_comparisons = comparisons.clone()
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)

for iteration in range(2):
    preference_model = PairwiseGP(history_X, history_comparisons)
    preference_model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    preference_model.eval()

    eubo = AnalyticExpectedUtilityOfBestOption(pref_model=preference_model)
    next_pair, eubo_value = optimize_acqf(
        acq_function=eubo,
        bounds=bounds,
        q=2,
        num_restarts=2,
        raw_samples=16,
        options={"maxiter": 20},
    )
    next_pair = next_pair.detach()
    values = latent_utility(next_pair)
    winner, loser = (0, 1) if values[0] >= values[1] else (1, 0)
    offset = len(history_X)
    history_X = torch.cat((history_X, next_pair), dim=0)
    new_comparison = torch.tensor([[offset + winner, offset + loser]], dtype=torch.long)
    history_comparisons = torch.cat((history_comparisons, new_comparison), dim=0)
    print(f"round {iteration + 1}: pair={next_pair.squeeze(-1).tolist()}, EUBO={eubo_value.item():.4f}")

final_model = PairwiseGP(history_X, history_comparisons)
final_model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
final_model.eval()
print("accumulated:", history_X.shape, history_comparisons.shape)

In [ ]:
print("initial items:", len(datapoints), "initial comparisons:", len(comparisons))
print("updated items:", len(history_X), "updated comparisons:", len(history_comparisons))
assert len(history_X) == len(datapoints) + 2 * 2
assert len(history_comparisons) == len(comparisons) + 2
assert history_comparisons.max().item() < len(history_X)

with torch.no_grad():
    updated_posterior = final_model.posterior(test_X)
    updated_mean = updated_posterior.mean.squeeze(-1)
plt.figure(figsize=(9, 4))
plt.plot(test_X.squeeze(-1), mean_centered, label="before (centered)")
plt.plot(test_X.squeeze(-1), updated_mean - updated_mean.mean(), label="after (centered)")
plt.plot(test_X.squeeze(-1), true_centered, "--", label="true (centered)")
plt.xlabel("x")
plt.ylabel("relative utility")
plt.title("Posterior mean before and after two comparisons")
plt.legend()
plt.show()

## 9. EUBOとランダム比較のベンチマーク

同じ初期データを使い、合成効用のSimple Regretを比較する実行可能なスクリプトは `examples/preference_eubo_benchmark.py` にあります。Notebookでは結果を断定せず、必要な試行回数と乱数シードを指定して評価してください。

### ベンチマークの読み方

1回の実行でEUBOがランダム比較より必ず優れるとは限りません。比較するなら、初期候補・比較回数・乱数シード・計算予算を揃え、複数シードのSimple Regretの分布を確認してください。

## まとめ

| 処理 | このNotebookのAPI |
|---|---|
| 比較データ | `datapoints` + `comparisons`（勝者・敗者の行番号） |
| モデル | `robotorchan.models.PairwiseGP` |
| 学習 | `model.fit()`（Pairwise Laplace MLL） |
| 潜在効用の推定 | `model.posterior(test_X)` |
| 比較相手を固定した探索 | `AnalyticExpectedUtilityOfBestOption(previous_winner=...)` + `optimize_acqf(q=1)` |
| 2候補同時提案 | `AnalyticExpectedUtilityOfBestOption` + `optimize_acqf(q=2)` |
| 観測の追加 | `history_X` と `history_comparisons` の更新 |
| 次の学習 | 更新済み履歴で `PairwiseGP(...).fit()` |

**通常の回帰とは異なり、観測するのは絶対的な目的関数値ではなく比較の勝敗です。** このNotebookは候補提案・比較結果の取得・再学習までを実行します。